# 🧪 StockPred-ML 크루원 빠른 실험 템플릿 (Quick Experiment)
이 노트북에서는 `stockpred` 패키지를 불러와 **커스텀 파생변수 추가**, **하이퍼파라미터 변경**, **백테스트 평가**를 대화형으로 즉시 실험할 수 있습니다.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from stockpred import DataLoader, FeatureEngine, ModelTrainer, Backtester, TrainingConfig

## 1. 시드 데이터 로드 및 기본 피처 확인

In [ ]:
raw_df, asset_name, ticker = DataLoader.get_stock_data('SnP500', use_yfinance=False)
print(f'{asset_name} ({ticker}): 총 {len(raw_df):,}건 로드 완료')
raw_df.tail()

## 2. 모델 학습 및 Out-of-Sample 성적 확인

In [ ]:
trainer = ModelTrainer.train(
    asset_key='SnP500',
    raw_df=raw_df,
    window_size=TrainingConfig.WINDOW_SIZE,
    threshold=TrainingConfig.THRESHOLD,
    fast_mode=True,
    save_artifacts=False
)
pd.DataFrame(trainer.eval_metrics).T

## 3. 최근 20거래일 백테스트 및 상위 피처 중요도 확인

In [ ]:
bt = Backtester.evaluate(trainer, lookback_days=20)
print(f"최근 20일 적중률: {bt['hit_ratio_pct']}% ({bt['hit_count']}/{bt['total_days']})")
pd.DataFrame(trainer.get_feature_importances('XGB', top_k=10))